# Supervised Fine-Tuning

Supervised Fine-Tuning (SFT) is a process primarily used to adapt pre-trained language models to follow instructions, engage in dialogue, and use specific output formats. While pre-trained models have impressive general capabilities, SFT helps transform them into assistant-like models that can better understand and respond to user prompts. This is typically done by training on datasets of human-written conversations and instructions.

This page provides a step-by-step guide to fine-tuning the [`deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B`](https://huggingface.co/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B) model using the [`SFTTrainer`](https://huggingface.co/docs/trl/en/sft_trainer). By following these steps, you can adapt the model to perform specific tasks more effectively.

## When to Use SFT

Before diving into implementation, it's important to understand when SFT is the right choice for your project. As a first step, you should consider whether using an existing instruction-tuned model with well-crafted prompts would suffice for your use case. SFT involves significant computational resources and engineering effort, so it should only be pursued when prompting existing models proves insufficient.

> **Tip:** Consider SFT only if you:
> - Need additional performance beyond what prompting can achieve
> - Have a specific use case where the cost of using a large general-purpose model outweighs the cost of fine-tuning a smaller model
> - Require specialized output formats or domain-specific knowledge that existing models struggle with

If you determine that SFT is necessary, the decision to proceed depends on two primary factors:

### Template Control

SFT allows precise control over the model's output structure. This is particularly valuable when you need the model to:
1. Generate responses in a specific chat template format
2. Follow strict output schemas
3. Maintain consistent styling across responses

### Domain Adaptation

When working in specialized domains, SFT helps align the model with domain-specific requirements by:
1. Teaching domain terminology and concepts
2. Enforcing professional standards
3. Handling technical queries appropriately
4. Following industry-specific guidelines

> **Tip:** Before starting SFT, evaluate whether your use case requires:
> - Precise output formatting
> - Domain-specific knowledge
> - Consistent response patterns
> - Adherence to specific guidelines
>
> This evaluation will help determine if SFT is the right approach for your needs.

## Dataset Preparation

The supervised fine-tuning process requires a task-specific dataset structured with input-output pairs. Each pair should consist of:
1. An input prompt
2. The expected model response
3. Any additional context or metadata

The quality of your training data is crucial for successful fine-tuning. Let's look at how to prepare and validate your dataset.

## Training Configuration

The success of your fine-tuning depends heavily on choosing the right training parameters. Let's explore each important parameter and how to configure them effectively:

The SFTTrainer configuration requires consideration of several parameters that control the training process. Let's explore each parameter and their purpose:

1. **Training Duration Parameters**:
   - `num_train_epochs`: Controls total training duration
   - `max_steps`: Alternative to epochs, sets maximum number of training steps
   - More epochs allow better learning but risk overfitting

2. **Batch Size Parameters**:
   - `per_device_train_batch_size`: Determines memory usage and training stability
   - `gradient_accumulation_steps`: Enables larger effective batch sizes
   - Larger batches provide more stable gradients but require more memory

3. **Learning Rate Parameters**:
   - `learning_rate`: Controls size of weight updates
   - `warmup_ratio`: Portion of training used for learning rate warmup
   - Too high can cause instability, too low results in slow learning

4. **Monitoring Parameters**:
   - `logging_steps`: Frequency of metric logging
   - `eval_steps`: How often to evaluate on validation data
   - `save_steps`: Frequency of model checkpoint saves

> **Tip:** Start with conservative values and adjust based on monitoring:
> - Begin with 1-3 epochs
> - Use smaller batch sizes initially
> - Monitor validation metrics closely
> - Adjust learning rate if training is unstable

## Implementation with TRL

Now that we understand the key components, let's implement the training with proper validation and monitoring. We will use the `SFTTrainer` class from the Transformers Reinforcement Learning (TRL) library, which is built on top of the `transformers` library. Here's a complete example using the TRL library:

In [ ]:
# ============================================================
# 导入必要的库
# ============================================================
from datasets import load_dataset                          # HuggingFace 数据集加载工具
from transformers import AutoModelForCausalLM, AutoTokenizer  # 自动加载因果语言模型和分词器
from trl import SFTConfig, SFTTrainer                      # TRL 库：监督微调配置与训练器
import torch
import os

# ============================================================
# 设备选择：优先使用 GPU（cuda），无 GPU 则回退到 CPU
# ============================================================
device = "cuda" if torch.cuda.is_available() else "cpu"

# ============================================================
# 加载数据集
# HuggingFaceTB/smoltalk 是一个多轮对话数据集，"all" 表示加载全部子集
# 数据集包含 "train" 和 "test" 两个分割，每条样本有 "messages" 字段
# messages 格式为：[{"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]
# ============================================================
dataset = load_dataset("HuggingFaceTB/smoltalk", "all")

# ============================================================
# 加载预训练模型和分词器
# SmolLM2-135M 是 HuggingFace 发布的 1.35 亿参数的小型语言模型（base 版本）
# .to(device) 将模型权重移动到指定设备（GPU/CPU）
# ============================================================
model_name = "HuggingFaceTB/SmolLM2-135M"
model = AutoModelForCausalLM.from_pretrained(pretrained_model_name_or_path=model_name).to(
    device
)
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_name)
# 告知 tokenizer 模型的最大上下文长度，避免超长序列产生警告
# tokenizer 默认会对超过此长度的序列发出警告，设置后警告消失
tokenizer.model_max_length = 8192

# ============================================================
# 手动设置 ChatML 对话模板
# 背景：旧版 TRL 提供 setup_chat_format() 函数来完成此操作，
#       但 TRL 0.8+ 已将其移除，需手动配置。
#
# ChatML 是一种通用的对话格式，结构如下：
#   <|im_start|>user
#   用户消息内容<|im_end|>
#   <|im_start|>assistant
#   模型回复内容<|im_end|>
#
# 模板使用 Jinja2 语法：
#   - 遍历 messages 列表，为每条消息添加 <|im_start|>role\ncontent<|im_end|> 包裹
#   - 若设置了 add_generation_prompt=True，则在末尾追加 <|im_start|>assistant\n
#     以提示模型开始生成回复
# ============================================================
CHATML_TEMPLATE = (
    "{% for message in messages %}"
    "{{'<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>' + '\n'}}"
    "{% endfor %}"
    "{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}"
)
tokenizer.chat_template = CHATML_TEMPLATE

# ============================================================
# 向分词器词表中添加 ChatML 特殊 token
# <|im_start|> 和 <|im_end|> 是 ChatML 格式的边界标记
# add_special_tokens() 返回实际新增的 token 数量
# 若有新增 token，需要调用 resize_token_embeddings() 扩展模型的
# Embedding 层，使新 token 获得可训练的向量表示
# ============================================================
special_tokens = {"additional_special_tokens": ["<|im_start|>", "<|im_end|>"]}
num_added = tokenizer.add_special_tokens(special_tokens)
if num_added > 0:
    model.resize_token_embeddings(len(tokenizer))

# ============================================================
# 配置 SFT 训练参数
# ============================================================
training_args = SFTConfig(
    output_dir="./sft_output",          # 模型 checkpoint 和日志的保存目录
    max_steps=1000,                      # 最大训练步数（与 num_train_epochs 二选一）
    per_device_train_batch_size=4,       # 每块 GPU/CPU 上的训练 batch 大小
    learning_rate=5e-5,                  # 初始学习率（微调常用范围：1e-5 ~ 5e-5）
    logging_steps=10,                    # 每隔 10 步记录一次训练指标（loss 等）
    save_steps=100,                      # 每隔 100 步保存一次模型 checkpoint
    eval_strategy="steps",              # 按步数触发验证（也可设为 "epoch"）
    eval_steps=50,                       # 每隔 50 步在验证集上评估一次
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
                                         # 启用 BF16 混合精度训练：显存减半、速度更快、数值比 FP16 更稳定
                                         # 需要 Ampere 架构以上的 GPU（如 A100、RTX 3090/4090）
                                         # 动态检测支持情况，CPU 或旧 GPU 上自动为 False
    max_length=8192,                     # TRL 0.29.x 中截断序列的参数名（旧版为 max_seq_length）
                                         # 对齐 SmolLM2-135M 的上下文窗口，避免超长样本导致索引越界
    dataset_num_proc=os.cpu_count(),     # 数据预处理（tokenization）使用的并行进程数
                                         # os.cpu_count() 自动获取 CPU 核心数以最大化速度
)

# ============================================================
# 初始化 SFTTrainer（监督微调训练器）
# SFTTrainer 会自动：
#   1. 根据 tokenizer.chat_template 将 messages 转换为模型输入
#   2. 处理 padding、截断等预处理操作
#   3. 管理训练循环、梯度更新、checkpoint 保存
# processing_class 传入 tokenizer，用于数据预处理
# ============================================================
trainer = SFTTrainer(
    model=model,                          # 待微调的模型
    args=training_args,                   # 上面定义的训练配置
    train_dataset=dataset["train"],       # 训练集
    eval_dataset=dataset["test"],         # 验证集（用于监控过拟合）
    processing_class=tokenizer,           # 分词器（用于将文本转换为 token ID）
)

# ============================================================
# 启动训练
# 训练过程中会按配置定期打印 loss、保存 checkpoint、评估验证集
# ============================================================
trainer.train()

> **Tip:** When using a dataset with a "messages" field (like the example above), the SFTTrainer automatically applies the model's chat template, which it retrieves from the hub. This means you don't need any additional configuration to handle chat-style conversations - the trainer will format the messages according to the model's expected template format.

## Packing the Dataset

The SFTTrainer supports example packing to optimize training efficiency. This feature allows multiple short examples to be packed into the same input sequence, maximizing GPU utilization during training. To enable packing, simply set `packing=True` in the SFTConfig constructor. When using packed datasets with `max_steps`, be aware that you may train for more epochs than expected depending on your packing configuration. You can customize how examples are combined using a formatting function - particularly useful when working with datasets that have multiple fields like question-answer pairs. For evaluation datasets, you can disable packing by setting `eval_packing=False` in the SFTConfig. Here's a basic example of customizing the packing configuration:

In [ ]:
# Configure packing
training_args = SFTConfig(packing=True)

trainer = SFTTrainer(model=model, train_dataset=dataset, args=training_args)

trainer.train()

When packing the dataset with multiple fields, you can define a custom formatting function to combine the fields into a single input sequence. This function should take a list of examples and return a dictionary with the packed input sequence. Here's an example of a custom formatting function:

In [ ]:
def formatting_func(example):
    text = f"### Question: {example['question']}\n ### Answer: {example['answer']}"
    return text

training_args = SFTConfig(packing=True)
trainer = SFTTrainer(
    "facebook/opt-350m",
    train_dataset=dataset,
    args=training_args,
    formatting_func=formatting_func,
)

## Monitoring Training Progress

Effective monitoring is crucial for successful fine-tuning. Let's explore what to watch for during training:

### Understanding Loss Patterns

Training loss typically follows three distinct phases:
1. **Initial Sharp Drop**: Rapid adaptation to new data distribution
2. **Gradual Stabilization**: Learning rate slows as model fine-tunes
3. **Convergence**: Loss values stabilize, indicating training completion

### Metrics to Monitor

Effective monitoring involves tracking quantitative metrics, and evaluating qualitative metrics. Available metrics are:

- Training loss
- Validation loss
- Learning rate progression
- Gradient norms

> **Warning:** Watch for these warning signs during training:
> 1. Validation loss increasing while training loss decreases (overfitting)
> 2. No significant improvement in loss values (underfitting)
> 3. Extremely low loss values (potential memorization)
> 4. Inconsistent output formatting (template learning issues)

### The Path to Convergence

As training progresses, the loss curve should gradually stabilize. The key indicator of healthy training is a small gap between training and validation loss, suggesting the model is learning generalizable patterns rather than memorizing specific examples. The absolute loss values will vary depending on your task and dataset.

### Monitoring Training Progress

A typical training progression shows both training and validation loss decreasing sharply at first, then gradually leveling off. This pattern indicates the model is learning effectively while maintaining generalization ability.

### Warning Signs to Watch For

Several patterns in the loss curves can indicate potential issues. Below we illustrate common warning signs and solutions that we can consider.

**Overfitting:** If the validation loss decreases at a significantly slower rate than training loss, your model is likely overfitting to the training data. Consider:
- Reducing the training steps
- Increasing the dataset size
- Validating dataset quality and diversity

**Underfitting:** If the loss doesn't show significant improvement, the model might be:
- Learning too slowly (try increasing the learning rate)
- Struggling with the task (check data quality and task complexity)
- Hitting architecture limitations (consider a different model)

**Memorization:** Extremely low loss values could suggest memorization rather than learning. This is particularly concerning if:
- The model performs poorly on new, similar examples
- The outputs lack diversity
- The responses are too similar to training examples

> **Warning:** Monitor both the loss values and the model's actual outputs during training. Sometimes the loss can look good while the model develops unwanted behaviors. Regular qualitative evaluation of the model's responses helps catch issues that metrics alone might miss.

We should note that the interpretation of the loss values we outline here is aimed on the most common case, and in fact, loss values can behave on various ways depending on the model, the dataset, the training parameters, etc. If you interested in exploring more about outlined patterns, you should check out this blog post by the people at [Fast AI](https://www.fast.ai/posts/2023-09-04-learning-jumps/).

## Evaluation after SFT

In section [11.4](https://huggingface.co/learn/llm-course/chapter11/4) we will learn how to evaluate the model using benchmark datasets. For now, we will focus on the qualitative evaluation of the model.

After completing SFT, consider these follow-up actions:

1. Evaluate the model thoroughly on held-out test data
2. Validate template adherence across various inputs
3. Test domain-specific knowledge retention
4. Monitor real-world performance metrics

> **Tip:** Document your training process, including:
> - Dataset characteristics
> - Training parameters
> - Performance metrics
> - Known limitations
>
> This documentation will be valuable for future model iterations.

## Quiz

### 1. What parameters control the training duration in SFT?

### 2. Which pattern in the loss curves indicates potential overfitting?

### 3. What is `gradient_accumulation_steps` used for?

### 4. What should you monitor during SFT training?

### 5. What indicates healthy convergence during training?

## 💐 Nice work!

You've learned how to fine-tune models using SFT! To continue your learning:
1. Try the notebook with different parameters
2. Experiment with other datasets
3. Contribute improvements to the course material

## Additional Resources

- [TRL Documentation](https://huggingface.co/docs/trl)
- [SFT Examples Repository](https://github.com/huggingface/trl/blob/main/trl/scripts/sft.py)
- [Fine-tuning Best Practices](https://huggingface.co/docs/transformers/training)